# $\mathrm{Na}_{\mathrm{V}}1.5$ main and supplemental figures

The main-text analysis uses **WT vanilla**, **QQQ vanilla**, and the **corrected QQQ
masked v1 protocol**. WT vanilla provides the native evolutionary baseline; QQQ
vanilla isolates the sequence substitution under the same MSA treatment; and QQQ
masked tests whether relaxing that evolutionary prior reveals additional
mutant-associated pore geometry.

Corrected masked v1 and v2 remain available as separate supplemental
mask-design comparisons. The unavailable no-IFM replacement is not substituted
from a retired cohort.

In [ ]:
from pathlib import Path
import sys
import importlib
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd() if (Path.cwd()/"shared").is_dir() else Path.cwd().parent
sys.path.insert(0,str(repo_root)) if str(repo_root) not in sys.path else None
import shared.nav15_presentation as nav15_presentation
importlib.reload(nav15_presentation)
from shared.data_access import resolve
from shared.dataset_selection import (
    distance_csv_options, load_selected_distance_csv, nav15_active_masked_distance_path,
)
from shared.plotting import NAV15_PALETTE, apply_kv21_style
from shared.nav15_pore_shape import experimental_gate_shape
from shared.nav15_presentation import (
    presentation_table, plot_main_mask_tradeoff, plot_selected_qqq_rmsd,
    plot_selected_qqq_reference_preference, plot_supplemental_mask_audit,
    plot_supplemental_pocket_profile,
)
apply_kv21_style()
data_dir=repo_root/'nav15'/'dataDistances'
DATASET_SELECTION='all_ok_3'
figure_dir=repo_root.parent/'vgci_mutants_writing'/'figures'/'nav15'
figure_dir.mkdir(parents=True,exist_ok=True)

In [ ]:
specs={
 'WT | vanilla':(distance_csv_options(repo_root, data_dir / '26-07-27_Nav15_wt_vanillaAF2_distances_all_all.csv', 'Nav15', 'WT', 'vanilla'),'WT',NAV15_PALETTE['WT_VAN']),
 'WT | masked v2':(distance_csv_options(repo_root, nav15_active_masked_distance_path(repo_root, 'WT', 'v2'), 'Nav15', 'WT', 'masked'),'WT',NAV15_PALETTE['WT_MASKED_V2']),
 'QQQ | vanilla':(distance_csv_options(repo_root, data_dir / '26-07-27_Nav15_qqq_vanilla_AF2_distances_all_all.csv', 'Nav15', 'QQQ', 'vanilla'),'QQQ',NAV15_PALETTE['QQQ_VAN']),
 'QQQ | masked':(distance_csv_options(repo_root, nav15_active_masked_distance_path(repo_root, 'QQQ', 'v1'), 'Nav15', 'QQQ', 'masked'),'QQQ',NAV15_PALETTE['QQQ_MASKED']),
 'QQQ | masked v2':(distance_csv_options(repo_root, nav15_active_masked_distance_path(repo_root, 'QQQ', 'v2'), 'Nav15', 'QQQ', 'masked'),'QQQ',NAV15_PALETTE['QQQ_MASKED_V2']),
}
datasets={label:(load_selected_distance_csv(label, options, DATASET_SELECTION),condition) for label,(options,condition,color) in specs.items()}
palette={label:color for label,(_,_,color) in specs.items()}
summary,models=presentation_table(datasets)
experimental_shape,_=experimental_gate_shape(repo_root/'nav15'/'experimental')
display(summary.round(3))

## Main Figure 1: mutation and selected-mask distance distributions

The three columns preserve the logic of the comparison. WT vanilla is the native
reference, QQQ vanilla shows the mutation with an intact MSA, and QQQ masked shows
the additional response after relaxing the evolutionary prior. The points over
the two pore panels are experimental structures; they are not additional model
ensembles.

In [ ]:
fig=plot_main_mask_tradeoff(summary,models,palette,experimental_shape)
fig.savefig(figure_dir/'Nav15_main_selected_mask_distances.png',dpi=600,bbox_inches='tight')
fig.savefig(figure_dir/'Nav15_main_selected_mask_distances.pdf',bbox_inches='tight')
plt.show()

## Main Figure 2: resemblance to the experimental QQQ/open pore

These panels compare QQQ vanilla directly with the corrected QQQ masked v1 cohort.
RMSD is calculated after alignment to the stable channel core. Lower values mean
that the modeled pore region more closely resembles the corresponding region of
7FBS. The DII-S6 panel is especially informative because it contains the
largest mask-associated improvement.

In [ ]:
rmsd_source=repo_root/'nav15'/'dataRMSD'/'Nav15_all_models_all_references_RMSD_distances_OK3.csv'
rmsd_columns=[
    'sequence_condition','protocol','reference_id','pdb_file',
    'pore_domain__ca__core_aligned_rmsd_A',
    'DII_s6__ca__core_aligned_rmsd_A',
]
if rmsd_source.is_file():
    rmsd=pd.read_csv(rmsd_source,usecols=rmsd_columns,low_memory=False)
    fig=plot_selected_qqq_rmsd(rmsd,palette)
    fig.savefig(figure_dir/'Nav15_main_QQQ_7FBS_RMSD.png',dpi=600,bbox_inches='tight')
    fig.savefig(figure_dir/'Nav15_main_QQQ_7FBS_RMSD.pdf',bbox_inches='tight')
    plt.show()
else:
    rmsd=None
    print(f'RMSD-to-reference input unavailable; skipped experimental-reference figure: {rmsd_source}')

## Main Figure 3: paired experimental-reference preference

For each model, the signed score is

$$\Delta_{7FBS-8VYJ}=RMSD_{7FBS}-RMSD_{8VYJ}.$$

Negative values indicate a pore closer to the engineered QQQ/open 7FBS
reference; positive values indicate a pore closer to the native-open 8VYJ
reference. This is a structural resemblance score, not a direct functional-state
assignment.

In [ ]:
from shared.rmsd_analysis import reference_preference
if rmsd is not None:
    preference=reference_preference(rmsd,'pore_domain__ca__core_aligned_rmsd_A','8VYJ','7FBS')
    fig=plot_selected_qqq_reference_preference(preference,palette)
    fig.savefig(figure_dir/'Nav15_main_QQQ_reference_preference.png',dpi=600,bbox_inches='tight')
    fig.savefig(figure_dir/'Nav15_main_QQQ_reference_preference.pdf',bbox_inches='tight')
    plt.show()
else:
    print('RMSD-to-reference input unavailable; skipped paired reference-preference figure.')

## Supplemental Figure S1: all mask designs

Corrected WT and QQQ masked v1/v2 profiles are retained here to show sensitivity to mask design.
The WT no-IFM replacement is not staged and is deliberately excluded rather than replaced by
a retired masked cohort.

In [ ]:
fig=plot_supplemental_mask_audit(summary,palette)
fig.savefig(figure_dir/'Nav15_supplement_mask_audit.png',dpi=300,bbox_inches='tight')
fig.savefig(figure_dir/'Nav15_supplement_mask_audit.pdf',bbox_inches='tight')
plt.show()

## Supplemental figure S2: receptor-pocket profile

The heatmap shows which receptor contacts are retained or lost rather than reducing engagement to a single binary label.

In [ ]:
pocket_table,fig=plot_supplemental_pocket_profile(datasets)
fig.savefig(figure_dir/'Nav15_supplement_pocket_profile.png',dpi=300,bbox_inches='tight')
fig.savefig(figure_dir/'Nav15_supplement_pocket_profile.pdf',bbox_inches='tight')
plt.show()
presentation_dir=repo_root/'nav15'/'dataRMSD'/'analysis'/'presentation'
presentation_dir.mkdir(parents=True,exist_ok=True)
summary.to_csv(presentation_dir/'Nav15_mask_presentation_summary.csv',index=False)